# 📧 Spam Detection AI - Model Analysis

This notebook provides a deep dive into the **Spam Detection AI** model. We will explore the dataset, visualise the class distribution, and evaluate the performance of our **Multinomial Naive Bayes** classifier.

### Tech Stack:
*   **Data Handling**: Pandas, Numpy
*   **Machine Learning**: Scikit-learn (TF-IDF + Naive Bayes)
*   **Visualisation**: Matplotlib, Seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Preprocessing helper (imported logic)
import string
from nltk.corpus import stopwords

# Ensure NLTK data is ready
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ Libraries Loaded Successfully")

## 1. Data Loading & Exploration
We load the **SMS Spam Collection** dataset. It contains 5,572 records with two columns: `label` (ham/spam) and `message`.

In [ ]:
# Path to the dataset
DATA_PATH = "ml_training/spam.csv"

# Load the dataset
df = pd.read_csv(DATA_PATH, encoding='latin-1', header=None)
df = df[[0, 1]]
df.columns = ['label', 'message']

# Clean empty rows
df.dropna(inplace=True)
df = df[df['label'].isin(['ham', 'spam'])]

print(f"Dataset Size: {df.shape[0]} rows")
df.head()

### 📊 2. Class Distribution
Visualising how many spam vs ham messages we have. The dataset is typically imbalanced (more ham than spam).

In [ ]:
plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")
ax = sns.countplot(data=df, x='label', hue='label', palette={'ham': '#16a34a', 'spam': '#ea580c'}, legend=False)
plt.title("Distribution of Spam vs Legitimate Messages", fontsize=14, fontweight='bold')
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

## 3. Data Preprocessing
Before training, we must clean the text by converting to lowercase, removing punctuation, and stripping away common stop words.

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = "".join([ch for ch in text if ch not in string.punctuation])
    stop_words = set(stopwords.words('english'))
    tokens = text.split()
    return " ".join([word for word in tokens if word not in stop_words])

df['clean_message'] = df['message'].apply(preprocess_text)
df[['message', 'clean_message']].head()

## 4. Model Training
We split the data (80% training, 20% testing) and build a **Pipeline** that combines:
1.  **TF-IDF Vectorizer**: To weight features.
2.  **Multinomial Naive Bayes**: To classify.

In [ ]:
X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])

pipeline.fit(X_train, y_train)
print("🚀 Model training complete!")

## 📊 5. Evaluation
We evaluate the model using a **Confusion Matrix** to see where the model makes errors.

In [ ]:
y_pred = pipeline.predict(X_test)

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%\n")
print(classification_report(y_test, y_pred))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Ham', 'Spam'], 
            yticklabels=['Ham', 'Spam'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 🤖 6. Live Testing
Run the cell below and enter your own message to see the AI in action!

In [ ]:
def predict_message(text):
    clean = preprocess_text(text)
    pred = pipeline.predict([clean])[0]
    proba = pipeline.predict_proba([clean])[0]
    confidence = max(proba) * 100
    
    color = "\033[91m" if pred == "spam" else "\033[92m"
    print(f"Message: {text}")
    print(f"Result:  {color}{pred.upper()}\033[0m ({confidence:.2f}% confidence)")
    print("-"*30)

# --- TEST HERE ---
user_msg = "WINNER!! You have been selected to receive a £1000 prize!"
predict_message(user_msg)

user_msg = "Hey, are we still meeting for lunch at noon?"
predict_message(user_msg)